# Contextual Encoding Model

Runs Ridge Regression encoding models using XLM-RoBERTa sliding window embeddings
to predict ECoG brain activity. Mirrors Eyal's static encoding pipeline exactly,
using the same `process_embeddings` function from `static_encoding.py`.

## Conditions tested
| Condition | language_mode | en_embedding | he_embedding | ar_embedding |
|-----------|--------------|--------------|--------------|------------------|
| en | en | English (768d) | Hebrew | Arabic |
| he | he | English | Hebrew (768d) | Arabic |
| ar | ar | English | Hebrew | Arabic (768d) |
| en+he_residual | en+he | English (768d) | Hebrew residual (768d) | Arabic |
| en+ar_residual | en+ar | English (768d) | Hebrew | Arabic residual (768d) |
| noise | noise | Noise (768d) | Noise | Noise |

## Key research questions
1. Does English contextual (XLM-RoBERTa) outperform English static (FastText, Eyal)?
2. Does English + Hebrew residual outperform English alone?
3. Does English + Arabic residual outperform English alone?
4. Do all conditions outperform noise?

## Requirements
- `static_encoding.py` from Eyal's codebase (must be in the same directory)
- ECoG `.fif` files in `./subject_data/`
- Sliding window embeddings from notebook 04
- Residuals from notebook 05

## 1. Imports

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('../data/Amirim_Project_Submission'))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import io
import warnings
from contextlib import redirect_stdout

import mne
from mne_bids import BIDSPath
from nilearn.plotting import plot_markers

# Reuse Eyal's encoding pipeline exactly — no modifications
from static_encoding import process_embeddings

print('Imports ready.')

Imports ready.


## 2. Load Embeddings and Residuals

In [2]:
DATA_DIR        = '../data/processed/'
WORD_LEVEL_PATH = '../data/Amirim_Project_Submission/translated_podcast_transcript_filtered.csv'

# ── EMBEDDING MODE ────────────────────────────────────────────────────────────
# Set this to choose which embeddings to use. Must match notebook 05.
#
#   'sliding_window'  — 1735 words, 32-word context window (notebook 04)
#   'contextual'      — 1692 words, full sentence context  (notebooks 01–03)
#
EMBEDDING_MODE = 'sliding_window'   # Change this to 'sliding_window' or 'contextual' as needed

full_transcript = pd.read_csv(WORD_LEVEL_PATH)
print(f'Embedding mode : {EMBEDDING_MODE}')

if EMBEDDING_MODE == 'sliding_window':
    E = pd.read_csv(DATA_DIR + 'en_sliding_window_embeddings.csv').values.astype(np.float32)
    H = pd.read_csv(DATA_DIR + 'he_sliding_window_embeddings.csv').values.astype(np.float32)
    A = pd.read_csv(DATA_DIR + 'ar_sliding_window_embeddings.csv').values.astype(np.float32)
    word_level_df = full_transcript[['start', 'end', 'en', 'he', 'ar']].reset_index(drop=True)

elif EMBEDDING_MODE == 'contextual':
    en_idx   = pd.read_csv(DATA_DIR + 'en_contextual_matched_indices.csv')
    orig_idx = en_idx['original_word_idx'].values          # 1692 indices into transcript
    E = pd.read_csv(DATA_DIR + 'en_contextual_aligned_embeddings.csv').values.astype(np.float32)
    H = pd.read_csv(DATA_DIR + 'he_contextual_aligned_embeddings.csv').values[orig_idx].astype(np.float32)
    A = pd.read_csv(DATA_DIR + 'ar_contextual_aligned_embeddings.csv').values[orig_idx].astype(np.float32)
    word_level_df = full_transcript.iloc[orig_idx][['start', 'end', 'en', 'he', 'ar']].reset_index(drop=True)

else:
    raise ValueError(f'Unknown EMBEDDING_MODE: {EMBEDDING_MODE!r}. Choose sliding_window or contextual.')

# Residuals — must be generated by notebook 05 with the same EMBEDDING_MODE
H_residual = np.load(DATA_DIR + f'hebrew_residuals_{EMBEDDING_MODE}.npy').astype(np.float32)
A_residual = np.load(DATA_DIR + f'arabic_residuals_{EMBEDDING_MODE}.npy').astype(np.float32)

# Noise control — same shape and distribution as English
rng   = np.random.RandomState(42)
NOISE = rng.normal(E.mean(), E.std(), size=E.shape).astype(np.float32)

print(f'Words          : {len(word_level_df)}')
print(f'English        : {E.shape}')
print(f'Hebrew         : {H.shape}')
print(f'Arabic         : {A.shape}')
print(f'Hebrew residual: {H_residual.shape}')
print(f'Arabic residual: {A_residual.shape}')
assert E.shape == H.shape == A.shape, 'Shape mismatch between embedding matrices!'
print('\nAll shapes verified ✓')

Embedding mode : sliding_window
Words          : 1735
English        : (1735, 768)
Hebrew         : (1735, 768)
Arabic         : (1735, 768)
Hebrew residual: (1735, 768)
Arabic residual: (1735, 768)

All shapes verified ✓


## 3. Build Conditions and Base DataFrame

In [3]:
# Base DataFrame with timing info — shared across all conditions
base_df = pd.DataFrame({
    'start': word_level_df['start'].values,
    'end'  : word_level_df['end'].values,
})

# Each condition specifies:
#   condition_name : used for output filenames
#   language_mode  : passed directly to process_embeddings
#                    'en'    -> uses en_embedding column only
#                    'he'    -> uses he_embedding column only
#                    'ar'    -> uses ar_embedding column only
#                    'en+he' -> concatenates en_embedding + he_embedding
#                    'en+ar' -> concatenates en_embedding + ar_embedding
#                    'noise' -> generates noise internally
#   en_mat, he_mat, ar_mat : what to store in each column
#
# For en+he_residual: language_mode='en+he' means process_embeddings
# concatenates en_embedding and he_embedding. We store the Hebrew residual
# in he_embedding so the result is EN (768d) + HE_RESIDUAL (768d) = 1536d.

CONDITIONS = [
    ('en',             'en',    E,      H,          A),
    ('he',             'he',    E,      H,          A),
    ('ar',             'ar',    E,      H,          A),
    ('en+he_residual', 'en+he', E,      H_residual, A),
    ('en+ar_residual', 'en+ar', E,      H,          A_residual),
    ('noise',          'noise', E,      H,          A),  # noise generated internally
]

print('Conditions to run:')
print(f'{"Name":22s}  {"Mode":10s}  {"en":12s}  {"he":12s}  {"ar"}')
print('-' * 75)
for name, mode, en, he, ar in CONDITIONS:
    print(f'{name:22s}  {mode:10s}  {str(en.shape):12s}  {str(he.shape):12s}  {ar.shape}')

Conditions to run:
Name                    Mode        en            he            ar
---------------------------------------------------------------------------
en                      en          (1735, 768)   (1735, 768)   (1735, 768)
he                      he          (1735, 768)   (1735, 768)   (1735, 768)
ar                      ar          (1735, 768)   (1735, 768)   (1735, 768)
en+he_residual          en+he       (1735, 768)   (1735, 768)   (1735, 768)
en+ar_residual          en+ar       (1735, 768)   (1735, 768)   (1735, 768)
noise                   noise       (1735, 768)   (1735, 768)   (1735, 768)


## 4. Config

In [4]:
import sys, json
from datetime import datetime
sys.path.append('../data/Amirim_Project_Submission/')

freq       = 64
tmin, tmax = -2.0, 2.0
use_PCA    = True
PCA_dim    = 150

datapath = '../data/ds005574/derivatives/ecogprep'

# Results go to:  encoding_results_64Hz_(-2.0,2.0) / <mode> /
# so every file's path makes the mode unambiguous.
outpath = f'./encoding_results_{freq}Hz_({tmin},{tmax})/{EMBEDDING_MODE}/'
os.makedirs(outpath, exist_ok=True)

subjects = [f'{i:02d}' for i in range(1, 10)]

# ── Write a config file so each results folder is self-documenting ────────────
run_config = {
    'EMBEDDING_MODE' : EMBEDDING_MODE,
    'freq'           : freq,
    'tmin'           : tmin,
    'tmax'           : tmax,
    'use_PCA'        : use_PCA,
    'PCA_dim'        : PCA_dim,
    'n_words'        : int(len(word_level_df)),
    'embedding_shape': list(E.shape),
    'subjects'       : subjects,
    'conditions'     : [c[0] for c in CONDITIONS],
    'run_timestamp'  : datetime.now().isoformat(timespec='seconds'),
}
with open(os.path.join(outpath, 'run_config.json'), 'w') as f:
    json.dump(run_config, f, indent=2)

print(f'Embedding mode : {EMBEDDING_MODE}')
print(f'Frequency      : {freq} Hz')
print(f'Time window    : {tmin}s to {tmax}s')
print(f'PCA            : {use_PCA}  (dim={PCA_dim})')
print(f'Words          : {len(word_level_df)}')
print(f'Subjects       : {subjects}')
print(f'Conditions     : {[c[0] for c in CONDITIONS]}')
print(f'Output dir     : {outpath}')
print(f'Total runs     : {len(subjects)} x {len(CONDITIONS)} = {len(subjects)*len(CONDITIONS)}')

Embedding mode : sliding_window
Frequency      : 64 Hz
Time window    : -2.0s to 2.0s
PCA            : True  (dim=150)
Words          : 1735
Subjects       : ['01', '02', '03', '04', '05', '06', '07', '08', '09']
Conditions     : ['en', 'he', 'ar', 'en+he_residual', 'en+ar_residual', 'noise']
Output dir     : ./encoding_results_64Hz_(-2.0,2.0)/sliding_window/
Total runs     : 9 x 6 = 54


## 5. Encoding Loop

In [5]:
for subj in subjects:
    # Load subject ECoG data
    file_path = BIDSPath(
        root=f'../data/ds005574/derivatives/ecogprep',
        subject=subj, task='podcast', datatype='ieeg',
        description='highgamma', suffix='ieeg', extension='.fif'
    ).fpath  # use .fpath instead of .basename to get the full path

    fif_path = str(file_path)
    if not os.path.exists(fif_path):
        print(f'Subject {subj}: file not found at {fif_path}, skipping.')
        continue

    raw = mne.io.read_raw_fif(fif_path, verbose=False)
    print(f'\nSubject {subj}: {len(raw.info["ch_names"])} channels loaded')

    for condition_name, language_mode, en_mat, he_mat, ar_mat in tqdm(
        CONDITIONS, desc=f'Subject {subj}'
    ):
        # Build embedding_df for this condition
        # Each embedding column contains a list of float32 arrays
        # process_embeddings selects columns based on language_mode
        embedding_df = base_df.copy()
        embedding_df['en_embedding'] = list(en_mat.astype(np.float32))
        embedding_df['he_embedding'] = list(he_mat.astype(np.float32))
        embedding_df['ar_embedding'] = list(ar_mat.astype(np.float32))

        # Run Eyal's encoding pipeline with no modifications
        f = io.StringIO()
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            with redirect_stdout(f):
                _, cv_scores = process_embeddings(
                    embedding_df=embedding_df,
                    raw=raw,
                    channel_names_regex='',
                    freq=freq,
                    tmin=tmin,
                    tmax=tmax,
                    language_mode=language_mode,
                    random_noise_mode='over all embeds',
                    use_PCA=use_PCA,
                    PCA_dim=PCA_dim
                )

        # Save cv_scores
        score_fname = os.path.join(
            outpath, f'corrs subj={subj} - {condition_name}.npy'
        )
        np.save(score_fname, cv_scores)

        # Plot correlation over time
        lags = np.arange(tmin * 512, tmax * 512, (512 / freq)) / 512
        mean = cv_scores.mean((0, 1))
        err  = cv_scores.std((0, 1)) / np.sqrt(np.prod(cv_scores.shape[:2]))

        fig, ax = plt.subplots()
        ax.plot(lags, mean, color='black')
        ax.fill_between(lags, mean - err, mean + err, alpha=0.1, color='black')
        ax.axvline(0, c=(.9, .9, .9), ls='--')
        ax.axhline(0, c=(.9, .9, .9), ls='--')
        ax.set_xlabel('lag (s)')
        ax.set_ylabel('encoding performance (r ± sem)')
        ax.set_title(f'Subject {subj} — {condition_name}')
        ax.set_ylim(-0.02, 0.05)
        fig.savefig(
            os.path.join(outpath,
                f'correlation time subj={subj} - {condition_name}.png'),
            dpi=600, bbox_inches='tight'
        )
        plt.close(fig)

        # Plot correlation over electrodes
        values = cv_scores.mean(0).max(-1)
        ch2loc = {ch['ch_name']: ch['loc'][:3] for ch in raw.info['chs']}
        coords = np.vstack([ch2loc[ch] for ch in raw.info['ch_names']]) * 1000
        order  = values.argsort()

        lzr = plot_markers(
            values[order], coords[order],
            node_size=30, display_mode='lzr',
            node_vmin=0, node_vmax=0.28,
            node_cmap='inferno_r', colorbar=True
        )
        lzr.title(f'Subject {subj} — {condition_name}', size=10)
        lzr.savefig(
            os.path.join(outpath,
                f'correlation electrodes subj={subj} - {condition_name}.png'),
            dpi=600, bbox_inches='tight'
        )
        plt.close()

print('\nAll encoding runs complete.')
print(f'Results saved to: {outpath}')


Subject 01: 99 channels loaded


Subject 01:   0%|          | 0/6 [00:00<?, ?it/s]Error processing line 1 of /Users/YAHLIZ/miniforge3/envs/language_project_env/lib/python3.10/site-packages/distutils-precedence.pth:

  Traceback (most recent call last):
    File "/Users/YAHLIZ/miniforge3/envs/language_project_env/lib/python3.10/site.py", line 195, in addpackage
      exec(line)
    File "<string>", line 1, in <module>
  ModuleNotFoundError: No module named '_distutils_hack'

Remainder of file ignored
Subject 01: 100%|██████████| 6/6 [01:10<00:00, 11.78s/it]



Subject 02: 90 channels loaded


Subject 02: 100%|██████████| 6/6 [00:58<00:00,  9.68s/it]



Subject 03: 235 channels loaded


...


Subject 08: 72 channels loaded


Subject 08: 100%|██████████| 6/6 [00:48<00:00,  8.06s/it]



Subject 09: 188 channels loaded


Subject 09: 100%|██████████| 6/6 [01:57<00:00, 19.61s/it]


All encoding runs complete.
Results saved to: ./encoding_results_64Hz_(-2.0,2.0)/sliding_window/


## 6. Results Summary

In [6]:
print('=' * 65)
print('ENCODING RESULTS SUMMARY — CONTEXTUAL EMBEDDINGS')
print('=' * 65)
print(f'{"Condition":22s}  {"Mean peak r":>12s}  {"Best subj r":>12s}')
print('-' * 52)

for condition_name, _, _, _, _ in CONDITIONS:
    subject_peaks = []
    for subj in subjects:
        score_fname = os.path.join(
            outpath, f'corrs subj={subj} - {condition_name}.npy'
        )
        if os.path.exists(score_fname):
            scores = np.load(score_fname)
            subject_peaks.append(scores.mean(0).max(-1).mean())

    if subject_peaks:
        mean_r = np.mean(subject_peaks)
        best_r = np.max(subject_peaks)
        print(f'{condition_name:22s}  {mean_r:12.4f}  {best_r:12.4f}')
    else:
        print(f'{condition_name:22s}  no results found')

print('\nInterpretation:')
print('  noise            should be near 0 (sanity check)')
print('  en               contextual English — compare to Eyal static baseline')
print('  en+he/ar_residual  residual conditions — do they beat English alone?')

ENCODING RESULTS SUMMARY — CONTEXTUAL EMBEDDINGS
Condition                Mean peak r   Best subj r
----------------------------------------------------
en                            0.0949        0.1032
he                            0.0969        0.1026
ar                            0.0958        0.1058
en+he_residual                0.0957        0.1018
en+ar_residual                0.0954        0.1029
noise                         0.0945        0.0966

Interpretation:
  noise            should be near 0 (sanity check)
  en               contextual English — compare to Eyal static baseline
  en+he/ar_residual  residual conditions — do they beat English alone?


In [7]:
# Load Eyal's saved scores for comparison
eyal_outpath = './encoding_results_64Hz_(-2.0,2.0)/'  # Eyal's output folder

print('Comparing contextual vs static:')
print(f'{"Condition":20s}  {"Contextual":>12s}  {"Static (Eyal)":>14s}')
print('-' * 52)

for condition_name in ['en', 'noise']:
    # Contextual
    ctx_peaks = []
    for subj in subjects:
        fname = os.path.join(outpath, f'corrs subj={subj} - {condition_name}.npy')
        if os.path.exists(fname):
            scores = np.load(fname)
            ctx_peaks.append(scores.mean(0).max(-1).mean())
    
    # Eyal's static
    static_peaks = []
    for subj in subjects:
        # Eyal's filename format
        if condition_name == 'en':
            eyal_fname = os.path.join(eyal_outpath, f'corrs subj={subj} - English.npy')
        else:
            eyal_fname = os.path.join(eyal_outpath, f'corrs subj={subj} - Noise over all embeds.npy')
        if os.path.exists(eyal_fname):
            scores = np.load(eyal_fname)
            static_peaks.append(scores.mean(0).max(-1).mean())
    
    ctx_mean   = np.mean(ctx_peaks)   if ctx_peaks   else float('nan')
    static_mean = np.mean(static_peaks) if static_peaks else float('nan')
    print(f'{condition_name:20s}  {ctx_mean:12.4f}  {static_mean:14.4f}')

Comparing contextual vs static:
Condition               Contextual   Static (Eyal)
----------------------------------------------------
en                          0.0949             nan
noise                       0.0945             nan
